# Chapter 5 — Reading Python Exceptions

**Book alignment:** Debugging AI From First Principles, Chapter 5

**Question this notebook isolates:** Two incidents end in a byte-identical `KeyError`
traceback. Does the two-pass reading (bottom-up to the raising frame, top-down to the first
diverging handoff) plus one old-snapshot swap separate a **code assumption** (Incident A —
fails on old data too) from a **data regression** (Incident B — passes on old data)?

In [ ]:
import traceback

def clean(orders, *, propagate_refund_id=True):
    out = []
    for o in orders:
        row = {"order_id": o["order_id"], "subtotal": o["subtotal"]}
        if propagate_refund_id and "refund_id" in o:
            row["refund_id"] = o["refund_id"]
        out.append(row)
    return out

def summarize(rows):
    total = 0.0
    for row in rows:
        total += row["subtotal"] - _refund(row)
    return round(total, 2)

def _refund(row):
    return 5.0 if row["refund_id"] else 0.0          # line that raises: KeyError 'refund_id'

def main(orders, *, code_version="A"):
    # version A never wrote the clean->summarize contract down; version B (fixed) does.
    rows = clean(orders, propagate_refund_id=(code_version == "B"))
    return summarize(rows)

CURRENT_INPUT   = [{"order_id": 1, "subtotal": 100.0, "refund_id": "RB-1"}]          # header lost the column
OLD_GOOD_INPUT  = [{"order_id": 1, "subtotal": 100.0, "refund_id": "RB-1"}]

## 1. Pass 1 — bottom-up: name the raising frame and confirm the operand

In [ ]:
try:
    main(CURRENT_INPUT, code_version="A")
except KeyError as e:
    tb = traceback.extract_tb(e.__traceback__)
    print("exception :", type(e).__name__, repr(e.args[0]))
    for fr in tb:
        print(f"  {fr.filename.split('/')[-1]}:{fr.lineno}  {fr.name}()   {fr.line}")
    raising = tb[-1]
    print(f"\nraising frame: {raising.name}()  <- where the program NOTICED")
assert raising.name == "_refund"
# confirm the operand, not the code text: the row really has no 'refund_id' key
row = clean(CURRENT_INPUT, propagate_refund_id=False)[0]
assert "refund_id" not in row
print("operand confirmed: row.keys() =", sorted(row))

## 2. Pass 2 — top-down: which handoff failed its contract?

`main` → `clean` → `summarize` → `_refund`. The contract "`clean` delivers `refund_id` on
every row" is what `summarize` assumed. The raising frame is `_refund`; the *diverging*
frame is the `clean → summarize` handoff.

In [ ]:
def handoff_ok(rows):
    return all("refund_id" in r for r in rows)

rows_A = clean(CURRENT_INPUT, propagate_refund_id=False)
print("clean -> summarize contract satisfied?", handoff_ok(rows_A))
assert not handoff_ok(rows_A)
print("first-diverging handoff: clean:NN -> summarize:NN  (the fix belongs HERE, not at _refund)")

## 3. The discriminating probe: same code x old known-good snapshot

In [ ]:
def outcome(inp, code_version):
    try:
        return main(inp, code_version=code_version)
    except KeyError:
        return "CRASH"

table = {
    ("A", "current input"):  outcome(CURRENT_INPUT, "A"),
    ("A", "old snapshot"):   outcome(OLD_GOOD_INPUT, "A"),   # A: code never guaranteed the key
}
for k, v in table.items():
    print(f"  code {k[0]}, {k[1]:14} -> {v}")

# Incident A (code assumption): old snapshot ALSO crashes -> Code layer
assert table[("A", "old snapshot")] == "CRASH"
# Incident B (data regression): the FIXED code (B) passes on the old snapshot -> Data layer
assert outcome(OLD_GOOD_INPUT, "B") != "CRASH" and outcome(CURRENT_INPUT, "B") != "CRASH"
print("\nA: crashes on old data too -> code-assumption. B (fixed contract): passes -> the earlier data loss was the regression.")
print("the traceback text is identical; only the swap separates them.")

## What we earned

The last line (`KeyError: 'refund_id'`) is a *mechanism*, not a diagnosis. Pass 1 (bottom-up)
names `_refund` as the raising frame and confirms the operand; Pass 2 (top-down) walks the
call chain and marks the `clean → summarize` handoff as the first contract violation — that
is where the fix belongs, not at the raising line. One old-snapshot swap then separates a
code assumption (still crashes) from a data regression (fixed code passes).

**Notebook 06 / Chapter 6** stops at that convicted handoff and inspects *live state* —
because the printed traceback never shows the values the code actually saw.